In [1]:
# =============================================================================
#  Dynamic Circuit Variant — Implementation A (Optional Task 3)
#  Probabilistic YK Decoder with Conditional Pauli Byproduct Correction
#
#  Two protocols compared on ibm_torino:
#
#  A) Post-selected (static):
#       Run circuit, measure (R,G), KEEP only shots where (crR=0, crG=0).
#       p_succ ≈ 0.25 → wastes 75% of shots.
#       No extra gates after measurement.
#
#  B) Dynamic (feedforward):
#       Run circuit, measure (R,G) MID-CIRCUIT, then ALWAYS apply the
#       correct Pauli byproduct correction to C before final readout:
#         crR=0,crG=0  →  I   (identity, no correction needed)
#         crR=1,crG=0  →  Z   on C
#         crR=0,crG=1  →  X   on C
#         crR=1,crG=1  →  XZ  on C  (i.e. Y up to phase)
#       Every shot is kept → 4× more efficient in shot cost.
#
#  Fidelity–shot-cost tradeoff:
#       For a target number of "useful" output samples N_useful, compare:
#         - Post-selected: need N_useful / p_succ total shots
#         - Dynamic:       need N_useful total shots  (all shots useful)
#
#  Backend  : ibm_torino (dynamic circuits confirmed available)
#  Qiskit   : if_test  (new-style control flow, Qiskit >= 1.0)
# =============================================================================

import numpy as np
import warnings, json, datetime
warnings.filterwarnings("ignore")
from collections import Counter, defaultdict
from scipy.stats import beta as beta_dist

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.quantum_info import Statevector, DensityMatrix, state_fidelity
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler, Batch

# =============================================================================
#  CONFIG
# =============================================================================
IBM_TOKEN   = "QfkScNfX4bVJ5lXm0082x7F7J6vya3SF5LJZFNOYXqqO"
BACKEND     = "ibm_torino"
SHOTS       = 10000
N_BOOTSTRAP = 2000
CI_LEVEL    = 0.95
SEED        = 42
RNG         = np.random.default_rng(SEED)
THETA_MSG   = 2.5349076035276403
VARPHI_MSG  = 2.0022404587009195

# =============================================================================
#  STEP 1 — Build the shared decoder core (no measurements, no corrections)
# =============================================================================
def build_decoder_core(qc, C, E, R, G, M, A, Y,
                       theta=THETA_MSG, varphi=VARPHI_MSG):
    """Append the full YK decoder onto an existing circuit up to Bell projection."""
    qc.u(theta, varphi, 0.0, M)
    qc.swap(C, M);  qc.barrier()
    qc.h(E);  qc.cx(E, M)
    qc.h(R);  qc.cx(R, G)
    qc.h(A);  qc.cx(A, Y);  qc.barrier()
    qc.cz(C, R);  qc.cz(E, R);  qc.cz(C, E)
    qc.h(C);  qc.h(E);  qc.h(R)
    qc.cz(C, R);  qc.cz(C, E);  qc.cz(E, R);  qc.barrier()
    qc.cz(A, G);  qc.cz(M, A);  qc.cz(G, M)
    qc.h(A);  qc.h(M);  qc.h(G)
    qc.cz(A, G);  qc.cz(G, M);  qc.cz(M, A);  qc.barrier()
    qc.cx(R, G);  qc.h(R);  qc.barrier()
    qc.swap(C, Y)
    # C now holds ρ_M (up to Pauli byproduct depending on (crR, crG))

# =============================================================================
#  STEP 2A — Post-selected circuits (3 Pauli bases, static)
# =============================================================================
def build_postselected_circuits():
    circuits = []
    for basis in ['Z', 'X', 'Y']:
        C=QuantumRegister(1,'C'); E=QuantumRegister(1,'E')
        R=QuantumRegister(1,'R'); G=QuantumRegister(1,'G')
        M=QuantumRegister(1,'M'); A=QuantumRegister(1,'A')
        Yr=QuantumRegister(1,'Y')
        crR=ClassicalRegister(1,'crR'); crG=ClassicalRegister(1,'crG')
        crT=ClassicalRegister(1,'crTomo')
        qc = QuantumCircuit(C,E,R,G,M,A,Yr,crR,crG,crT)
        build_decoder_core(qc,C,E,R,G,M,A,Yr)
        qc.measure(R, crR); qc.measure(G, crG)
        # Tomography rotation on C (no Pauli correction — discard non-00 shots)
        if basis == 'X': qc.h(C)
        elif basis == 'Y': qc.sdg(C); qc.h(C)
        qc.measure(C, crT)
        circuits.append((f'ps_{basis}', qc))
    return circuits

# =============================================================================
#  STEP 2B — Dynamic circuits (3 Pauli bases, with feedforward correction)
#
#  Byproduct correction table (standard teleportation result):
#    (crR, crG) = (0,0) → I     → no gate
#    (crR, crG) = (1,0) → Z     → qc.z(C)
#    (crR, crG) = (0,1) → X     → qc.x(C)
#    (crR, crG) = (1,1) → XZ    → qc.z(C); qc.x(C)
#
#  Implemented using Qiskit if_test blocks (new-style, Qiskit >= 1.0).
# =============================================================================
def build_dynamic_circuits():
    circuits = []
    for basis in ['Z', 'X', 'Y']:
        C=QuantumRegister(1,'C'); E=QuantumRegister(1,'E')
        R=QuantumRegister(1,'R'); G=QuantumRegister(1,'G')
        M=QuantumRegister(1,'M'); A=QuantumRegister(1,'A')
        Yr=QuantumRegister(1,'Y')
        crR=ClassicalRegister(1,'crR'); crG=ClassicalRegister(1,'crG')
        crT=ClassicalRegister(1,'crTomo')
        qc = QuantumCircuit(C,E,R,G,M,A,Yr,crR,crG,crT)
        build_decoder_core(qc,C,E,R,G,M,A,Yr)

        # Mid-circuit measurement of Bell outcome
        qc.measure(R, crR)
        qc.measure(G, crG)

        # Conditional Pauli byproduct corrections via feedforward
        # crR=1 → Z correction on C
        with qc.if_test((crR, 1)):
            qc.z(C)
        # crG=1 → X correction on C
        with qc.if_test((crG, 1)):
            qc.x(C)
        # After correction, C holds ρ_M regardless of (crR,crG) outcome

        qc.barrier()
        # Tomography rotation
        if basis == 'X': qc.h(C)
        elif basis == 'Y': qc.sdg(C); qc.h(C)
        qc.measure(C, crT)
        circuits.append((f'dyn_{basis}', qc))
    return circuits

# =============================================================================
#  STEP 3 — Connect and transpile
# =============================================================================
print("=" * 65)
print("  Dynamic Circuit Variant — ibm_torino")
print("=" * 65)

service = QiskitRuntimeService(channel="ibm_quantum_platform", token=IBM_TOKEN)
backend = service.backend(BACKEND)
print(f"\n[✓] Connected to {BACKEND}")

all_circuits_raw = build_postselected_circuits() + build_dynamic_circuits()

print(f"\n  Transpiling {len(all_circuits_raw)} circuits ...")
all_circuits_transpiled = []
for label, qc in all_circuits_raw:
    t = transpile(qc, backend=backend,
                  optimization_level=3,
                  seed_transpiler=SEED)
    depth  = t.depth()
    twoq   = sum(1 for _,qa,_ in t.data if len(qa) == 2)
    print(f"    {label:12s}: depth={depth:4d}  2q-gates={twoq:3d}")
    all_circuits_transpiled.append((label, t))

# =============================================================================
#  STEP 4 — Submit all 6 circuits in one Batch
# =============================================================================
print(f"\n  Submitting 6 circuits × {SHOTS} shots (Batch mode) ...")

jobs = {}
with Batch(backend=backend) as batch:
    sampler = Sampler(mode=batch)
    for label, t in all_circuits_transpiled:
        jobs[label] = sampler.run([t], shots=SHOTS)
        print(f"    {label}: job_id = {jobs[label].job_id()}")

# =============================================================================
#  STEP 5 — Collect results
# =============================================================================
def extract_counts(result):
    pub  = result[0]; data = pub.data
    crT  = data.crTomo.array.flatten()
    crG  = data.crG.array.flatten()
    crR  = data.crR.array.flatten()
    joint = [f"{int(t)}{int(g)}{int(r)}"
             for t,g,r in zip(crT, crG, crR)]
    return dict(Counter(joint))

raw = {}
print("\n  Collecting results ...")
for label, _ in all_circuits_transpiled:
    print(f"    Waiting for {label} ...", end=" ", flush=True)
    result     = jobs[label].result()
    raw[label] = extract_counts(result)
    total      = sum(raw[label].values())
    try:
        qs = jobs[label].metrics()['usage']['quantum_seconds']
        print(f"done ({total} counts, QPU={qs:.4f}s)")
    except Exception:
        print(f"done ({total} counts)")

# =============================================================================
#  STEP 6 — Post-process both protocols
# =============================================================================

# ── Helpers ───────────────────────────────────────────────────────────────────
def parse_bs(bs):
    p = bs.split(' ') if ' ' in bs else list(bs)
    return p[0], p[1], p[2]     # crTomo, crG, crR

def postselect_00(counts):
    """Keep only (crR=0, crG=0) shots. Returns tomo_counts, n_total, n_kept."""
    kept=defaultdict(int); nt=0; nk=0
    for bs,cnt in counts.items():
        nt += cnt
        crT,crG,crR = parse_bs(bs)
        if crR=='0' and crG=='0':
            kept[crT] += cnt; nk += cnt
    return dict(kept), nt, nk

def keep_all(counts):
    """For dynamic: all shots kept. Returns tomo_counts, n_total, n_kept."""
    kept=defaultdict(int); nt=0
    for bs,cnt in counts.items():
        nt += cnt
        crT,crG,crR = parse_bs(bs)
        kept[crT] += cnt
    return dict(kept), nt, nt

def pauli_exp(c):
    n0=c.get('0',0); n1=c.get('1',0); N=n0+n1
    return (n0-n1)/N if N>0 else 0.0

def reconstruct(sx,sy,sz):
    X=np.array([[0,1],[1,0]],dtype=complex)
    Y=np.array([[0,-1j],[1j,0]],dtype=complex)
    Z=np.array([[1,0],[0,-1]],dtype=complex)
    rho=(np.eye(2)+sx*X+sy*Y+sz*Z)/2
    ev,evec=np.linalg.eigh(rho)
    ev=np.maximum(ev,0); ev/=ev.sum()
    return (evec*ev)@evec.conj().T

def cp_ci(k, n, alpha=1-CI_LEVEL):
    lo = beta_dist.ppf(alpha/2,   k,   n-k+1) if k > 0 else 0.0
    hi = beta_dist.ppf(1-alpha/2, k+1, n-k  ) if k < n else 1.0
    return lo, hi

def bootstrap_F(tomo_dict_X, tomo_dict_Y, tomo_dict_Z, rho_M, n_boot=N_BOOTSTRAP):
    bsF = np.zeros(n_boot)
    for i in range(n_boot):
        def resample(d):
            keys=list(d.keys()); vals=np.array([d[k] for k in keys])
            N=vals.sum(); new=RNG.multinomial(N, vals/N)
            return {k:int(v) for k,v in zip(keys,new)}
        sx_b=pauli_exp(resample(tomo_dict_X))
        sy_b=pauli_exp(resample(tomo_dict_Y))
        sz_b=pauli_exp(resample(tomo_dict_Z))
        bsF[i]=state_fidelity(DensityMatrix(reconstruct(sx_b,sy_b,sz_b)), rho_M)
    return bsF

# ── Ideal target ─────────────────────────────────────────────────────────────
from qiskit import QuantumCircuit as QC
qcm=QC(1); qcm.u(THETA_MSG,VARPHI_MSG,0.0,0)
rho_M = DensityMatrix(Statevector.from_instruction(qcm))

# ── Post-selected protocol ────────────────────────────────────────────────────
ps_tomo = {}; ps_nkept = {}; ps_ntotal = {}
for basis in ['Z','X','Y']:
    kept, nt, nk = postselect_00(raw[f'ps_{basis}'])
    ps_tomo[basis]=kept; ps_ntotal[basis]=nt; ps_nkept[basis]=nk

p_succ_hat = ps_nkept['Z'] / ps_ntotal['Z']
cp_lo_ps, cp_hi_ps = cp_ci(ps_nkept['Z'], ps_ntotal['Z'])

sx_ps=pauli_exp(ps_tomo['X']); sy_ps=pauli_exp(ps_tomo['Y']); sz_ps=pauli_exp(ps_tomo['Z'])
rho_ps  = reconstruct(sx_ps, sy_ps, sz_ps)
F_ps    = float(state_fidelity(DensityMatrix(rho_ps), rho_M))
bsF_ps  = bootstrap_F(ps_tomo['X'], ps_tomo['Y'], ps_tomo['Z'], rho_M)
F_ps_lo = float(np.percentile(bsF_ps, 2.5))
F_ps_hi = float(np.percentile(bsF_ps, 97.5))

# ── Dynamic protocol ──────────────────────────────────────────────────────────
dyn_tomo = {}; dyn_nkept = {}; dyn_ntotal = {}
for basis in ['Z','X','Y']:
    kept, nt, nk = keep_all(raw[f'dyn_{basis}'])
    dyn_tomo[basis]=kept; dyn_ntotal[basis]=nt; dyn_nkept[basis]=nk

sx_dyn=pauli_exp(dyn_tomo['X']); sy_dyn=pauli_exp(dyn_tomo['Y']); sz_dyn=pauli_exp(dyn_tomo['Z'])
rho_dyn  = reconstruct(sx_dyn, sy_dyn, sz_dyn)
F_dyn    = float(state_fidelity(DensityMatrix(rho_dyn), rho_M))
bsF_dyn  = bootstrap_F(dyn_tomo['X'], dyn_tomo['Y'], dyn_tomo['Z'], rho_M)
F_dyn_lo = float(np.percentile(bsF_dyn, 2.5))
F_dyn_hi = float(np.percentile(bsF_dyn, 97.5))

# ── Fidelity–shot-cost tradeoff ────────────────────────────────────────────────
# For a target of N_useful post-selected output samples:
#   Post-selected needs: N_useful / p_succ  total shots
#   Dynamic needs:       N_useful            total shots
# Cost ratio = 1 / p_succ
cost_ratio = 1.0 / p_succ_hat

# =============================================================================
#  STEP 7 — Print paper-ready summary
# =============================================================================
print("\n" + "=" * 65)
print("  RESULTS SUMMARY — Dynamic vs Post-selected")
print("=" * 65)
print(f"""
  Backend          : {BACKEND}
  Shots per basis  : {SHOTS:,}

  ┌─────────────────────────────────────────────────────────┐
  │                   POST-SELECTED (static)                │
  ├─────────────────────────────────────────────────────────┤
  │  p_succ         : {p_succ_hat:.4f}  [{cp_lo_ps:.4f}, {cp_hi_ps:.4f}]  (CP 95% CI) │
  │  Shots kept     : {ps_nkept['Z']:,} / {ps_ntotal['Z']:,}  (Z basis)               │
  │  F(ρ_Y, ρ_M)    : {F_ps:.4f}  [{F_ps_lo:.4f}, {F_ps_hi:.4f}]  (bootstrap)  │
  │  Pauli <X,Y,Z>  : {sx_ps:+.4f}  {sy_ps:+.4f}  {sz_ps:+.4f}              │
  ├─────────────────────────────────────────────────────────┤
  │                   DYNAMIC (feedforward)                 │
  ├─────────────────────────────────────────────────────────┤
  │  Shots kept     : {dyn_nkept['Z']:,} / {dyn_ntotal['Z']:,}  (ALL shots useful)     │
  │  F(ρ_Y, ρ_M)    : {F_dyn:.4f}  [{F_dyn_lo:.4f}, {F_dyn_hi:.4f}]  (bootstrap)  │
  │  Pauli <X,Y,Z>  : {sx_dyn:+.4f}  {sy_dyn:+.4f}  {sz_dyn:+.4f}              │
  ├─────────────────────────────────────────────────────────┤
  │                   TRADEOFF                              │
  ├─────────────────────────────────────────────────────────┤
  │  Shot-cost ratio: {cost_ratio:.2f}×  (dynamic needs {cost_ratio:.2f}× fewer shots)  │
  │  ΔF (dyn - ps)  : {F_dyn - F_ps:+.4f}  (positive = dynamic is better)   │
  │  Fidelity cost  : {abs(F_dyn - F_ps):.4f}  from feedforward gate overhead         │
  └─────────────────────────────────────────────────────────┘
""")

# =============================================================================
#  STEP 8 — Save results
# =============================================================================
class _Enc(json.JSONEncoder):
    def default(self, o):
        if isinstance(o, (np.integer,)): return int(o)
        if isinstance(o, (np.floating,)): return float(o)
        if isinstance(o, np.ndarray): return o.tolist()
        if isinstance(o, complex): return {"re": o.real, "im": o.imag}
        return super().default(o)

def rho2json(r):
    return [[{"re": float(r[i,j].real), "im": float(r[i,j].imag)}
             for j in range(2)] for i in range(2)]

payload = {
    "metadata": {
        "backend": BACKEND, "shots": SHOTS,
        "timestamp": datetime.datetime.utcnow().isoformat()+"Z",
        "theta_msg": THETA_MSG, "varphi_msg": VARPHI_MSG,
    },
    "postselected": {
        "p_succ": float(p_succ_hat),
        "p_succ_ci": [float(cp_lo_ps), float(cp_hi_ps)],
        "n_kept": int(ps_nkept['Z']), "n_total": int(ps_ntotal['Z']),
        "bloch": {"sx": float(sx_ps), "sy": float(sy_ps), "sz": float(sz_ps)},
        "rho_Y": rho2json(rho_ps),
        "fidelity": float(F_ps),
        "fidelity_ci": [float(F_ps_lo), float(F_ps_hi)],
        "bootstrap_F": [float(v) for v in bsF_ps],
        "raw_counts": raw,
    },
    "dynamic": {
        "n_kept": int(dyn_nkept['Z']), "n_total": int(dyn_ntotal['Z']),
        "bloch": {"sx": float(sx_dyn), "sy": float(sy_dyn), "sz": float(sz_dyn)},
        "rho_Y": rho2json(rho_dyn),
        "fidelity": float(F_dyn),
        "fidelity_ci": [float(F_dyn_lo), float(F_dyn_hi)],
        "bootstrap_F": [float(v) for v in bsF_dyn],
    },
    "tradeoff": {
        "shot_cost_ratio": float(cost_ratio),
        "delta_F": float(F_dyn - F_ps),
        "description": (
            f"Dynamic circuit uses {cost_ratio:.2f}x fewer shots for same "
            f"useful output count, at a fidelity cost of {abs(F_dyn-F_ps):.4f}"
        )
    }
}

with open("dynamic_circuit_results.json", "w") as f:
    json.dump(payload, f, indent=2, cls=_Enc)
print("[✓] Results saved to dynamic_circuit_results.json")
print("[✓] Run plot_dynamic_comparison.py to generate paper figures.")

qiskit_runtime_service._discover_account:WARNING:2026-03-19 17:24:37,252: Loading account with the given token. A saved account will not be used.


  Dynamic Circuit Variant — ibm_torino


qiskit_runtime_service.__init__:WARNING:2026-03-19 17:24:40,869: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: CTCs. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().
qiskit_runtime_service.backends:WARNING:2026-03-19 17:24:40,869: Using instance: CTCs, plan: open



[✓] Connected to ibm_torino

  Transpiling 6 circuits ...
    ps_Z        : depth=  82  2q-gates= 32
    ps_X        : depth=  87  2q-gates= 32
    ps_Y        : depth=  86  2q-gates= 32
    dyn_Z       : depth=  87  2q-gates= 32
    dyn_X       : depth=  90  2q-gates= 32
    dyn_Y       : depth=  89  2q-gates= 32

  Submitting 6 circuits × 10000 shots (Batch mode) ...
    ps_Z: job_id = d6u6j6if84ks73ddu32g
    ps_X: job_id = d6u6j6ov5rlc73f3jpag
    ps_Y: job_id = d6u6j6ov5rlc73f3jpbg
    dyn_Z: job_id = d6u6j7469uic73chte70
    dyn_X: job_id = d6u6j72f84ks73ddu340
    dyn_Y: job_id = d6u6j78v5rlc73f3jpcg

    Waiting for ps_Z ... done (10000 counts, QPU=4.0000s)
    Waiting for ps_X ... done (10000 counts, QPU=4.0000s)
    Waiting for ps_Y ... done (10000 counts, QPU=4.0000s)
    Waiting for dyn_Z ... done (10000 counts, QPU=4.0000s)
    Waiting for dyn_X ... done (10000 counts, QPU=4.0000s)
    Waiting for dyn_Y ... done (10000 counts, QPU=4.0000s)

  RESULTS SUMMARY — Dynamic vs 